In [42]:
import os

In [43]:
benchmark_name = "c7552"
outLoad = 6.8008 ## used in hspice and sysC
RelStep = 3e6    ## used in hspice and sysC
reltotaltime = 3.6e7
worstPathPI = "N6"
worstPathPO = "N23"
vdd1 = 1.1
temp = 25
t = 50
resolution = ".1ps"
simTime = '10ns'
input_transition = 0.01 #used in generation of sysC TB

In [44]:
## verilog preprocess:
file_path = benchmark_name+".v"
preVer = []
with open(file_path, "r") as file:
    for line in file:
        if "not" in line:
            outp = (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" ")
            inp = (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" ")
            line = line.replace("not ", "nand ").replace("NOT1_","NAND2_rep").replace(inp+");", inp+", "+inp+");")
            preVer.append(line)
        else:
            preVer.append(line)

directory_path = benchmark_name+"/Verilog/"
os.makedirs(directory_path, exist_ok=True)
Ver_path = os.path.join(directory_path+benchmark_name+".v")
with open(Ver_path, "w") as file:
    file.writelines(preVer)
file.close()


In [45]:
## One input gates
class gate1:
    def __init__(self, name, numOfInp, outputZN, inputA1, gateNum):
        self.name = name
        self.numOfInp = numOfInp
        self.gateNum = gateNum
        self.inputA1 = inputA1
        self.outputZN = outputZN
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n A1 = {self.inputA1}\n ZN = {self.outputZN}"


In [46]:
## Two input gates
class gate2:
    def __init__(self, name, numOfInp, outputZN, inputA1, inputA2, gateNum):
        self.name = name
        self.numOfInp = numOfInp
        self.gateNum = gateNum
        self.inputA1 = inputA1
        self.inputA2 = inputA2
        self.outputZN = outputZN
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n A1 = {self.inputA1}\n A2 = {self.inputA2}\n ZN = {self.outputZN}"


In [47]:
## Three input gates
class gate3:
    def __init__(self, name, numOfInp, outputZN, inputA1, inputA2, inputA3, gateNum):
        self.name = name
        self.numOfInp = numOfInp
        self.gateNum = gateNum
        self.inputA1 = inputA1
        self.inputA2 = inputA2
        self.inputA3 = inputA3
        self.outputZN = outputZN
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n A1 = {self.inputA1}\n A2 = {self.inputA2}\n A3 = {self.inputA3}\n ZN = {self.outputZN}"


In [48]:
## Four input gates
class gate4:
    def __init__(self, name, numOfInp, outputZN, inputA1, inputA2, inputA3, inputA4, gateNum):
        self.name = name
        self.gateNum = gateNum
        self.numOfInp = numOfInp
        self.inputA1 = inputA1
        self.inputA2 = inputA2
        self.inputA3 = inputA3
        self.inputA4 = inputA4
        self.outputZN = outputZN
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n A1 = {self.inputA1}\n A2 = {self.inputA2}\n A3 = {self.inputA3}\n A4 = {self.inputA4}\n ZN = {self.outputZN}"


In [49]:
directory_path = benchmark_name+"/Verilog/"
file_path = os.path.join(directory_path+benchmark_name+".v")
i = 0
j = 0
net = []
required_gates = []
newGate = False
PIs = []
numOfPI = 0

wires=[]
wire_lines=""
with open(file_path, "r") as file:
    for line in file:
        if 'wire' in line:
            if(';' in line):
                wire_lines += line.rstrip('\n').split('wire')[1]
            else:
                wire_lines += line.rstrip('\n').split('wire')[1]
                while not(';' in line):
                    line = file.readline()  
                    wire_lines += line.rstrip('\n')
        wires = wire_lines
wires = wires.replace(" ", "").replace(";",",").split(',')
if (wires[len(wires)-1] == ""):
    wires = wires[:-1]
else: 
    wires[len(wires)-1] = wires[len(wires)-1].strip(',')
numOfwire = len(wires)
file.close()


with open(file_path, "r") as file:
    for line in file:
        if 'input' in line:
            PI_lines = line.rstrip('\n').split('input')[1]
            ## handling enters
            while ';' not in line:
                line = file.readline()
                PI_lines += line.rstrip('\n')
            numOfPI = len(PI_lines.split()[1:])
            print(PI_lines)
            PIs = PI_lines.replace(" ", "").strip(';').split(',')

        elif 'output' in line:
            PO_lines = line.rstrip('\n').split('output')[1]
            while ';' not in line:
                line = file.readline()
                PO_lines += line.rstrip('\n')
            numOfPO = len(PO_lines.split()[1:])
            POs = PO_lines.replace(" ", "").strip(';').split(',')
            
        elif 'nand' in line:
            newGate = True
            i=i+1
            currGateName = line.split()[1][0:5]
            if (line.split()[1][0:5] not in required_gates):
                required_gates.append(line.split()[1][0:5])
            numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
            # if (numOfGateInput == 1):
            #     net.append(gate1(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), i))
            # if (numOfGateInput == 2):
            #     net.append(gate2(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), i))
            # if (numOfGateInput == 3):
            #     net.append(gate3(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[3].strip(" "), i))
        
        elif 'nor' in line:
            newGate = True
            i=i+1
            currGateName = line.split()[1][0:4]
            if (line.split()[1][0:4] not in required_gates):
                required_gates.append(line.split()[1][0:4])
            numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
            # if (numOfGateInput == 2):
            #     net.append(gate2(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), i))
            # if (numOfGateInput == 3):
            #     net.append(gate3(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[3].strip(" "), i))
        
        elif 'not' in line:
            newGate = True
            i=i+1
            currGateName = line.split()[1][0:4]
            currGateName = "INV"
            if (currGateName not in required_gates):
                required_gates.append(currGateName)
            numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1

        elif ' or' in line:
            print(line)
            newGate = True
            i=i+1
            numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
            currGateName = "OR"+str(numOfGateInput)
            # currGateName = line.split()[1][0:3]
            if (currGateName not in required_gates):
                required_gates.append(currGateName)
            # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1


        elif 'and' in line:
            newGate = True
            i=i+1
            numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
            currGateName = "AND"+str(numOfGateInput)
            if (currGateName not in required_gates):
                required_gates.append(currGateName)
        else:
            newGate = False

        if (newGate):
            if (numOfGateInput == 1):
                net.append(gate1(currGateName, 1, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), i))
            elif (numOfGateInput == 2):
                net.append(gate2(currGateName, 2,(line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), i))
            elif (numOfGateInput == 3):
                net.append(gate3(currGateName, 3, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[3].strip(" "), i))
            elif (numOfGateInput == 4):
                net.append(gate4(currGateName, 4, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[3].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[4].strip(" "), i))
file.close()

 N1, N5, N9, N12, N15, N18, N23, N26, N29, N32, N35, N38, N41,       N44, N47, N50, N53, N54, N55, N56, N57, N58, N59, N60, N61, N62,       N63, N64, N65, N66, N69, N70, N73, N74, N75, N76, N77, N78, N79,       N80, N81, N82, N83, N84, N85, N86, N87, N88, N89, N94, N97,       N100, N103, N106, N109, N110, N111, N112, N113, N114, N115,       N118, N121, N124, N127, N130, N133, N134, N135, N138, N141,       N144, N147, N150, N151, N152, N153, N154, N155, N156, N157,       N158, N159, N160, N161, N162, N163, N164, N165, N166, N167,       N168, N169, N170, N171, N172, N173, N174, N175, N176, N177,       N178, N179, N180, N181, N182, N183, N184, N185, N186, N187,       N188, N189, N190, N191, N192, N193, N194, N195, N196, N197,       N198, N199, N200, N201, N202, N203, N204, N205, N206, N207,       N208, N209, N210, N211, N212, N213, N214, N215, N216, N217,       N218, N219, N220, N221, N222, N223, N224, N225, N226, N227,       N228, N229, N230, N231, N232, N233, N234, N235, N236, N237,    

In [50]:
required_gates

['AND2', 'AND3', 'AND4', 'NAND2', 'NOR2', 'NOR3', 'NOR4', 'OR2', 'OR3', 'OR4']

In [51]:
# num of gates
len(net)

2331

In [11]:
POs

['G1324',
 'G1325',
 'G1326',
 'G1327',
 'G1328',
 'G1329',
 'G1330',
 'G1331',
 'G1332',
 'G1333',
 'G1334',
 'G1335',
 'G1336',
 'G1337',
 'G1338',
 'G1339',
 'G1340',
 'G1341',
 'G1342',
 'G1343',
 'G1344',
 'G1345',
 'G1346',
 'G1347',
 'G1348',
 'G1349',
 'G1350',
 'G1351',
 'G1352',
 'G1353',
 'G1354',
 'G1355']

In [12]:
## gate HSPICE description
gate_model = []
Nangate_path = "stdcells.cdl"
## GOTTA CHANGE THEEEEEEESE :
gate_model.append(r".include '45nm\stdcells.cdl'")
gate_model.append("\n")
gate_model.append(r".include '45nm\models\hspice\tran_models\models_nom\PMOS_VTL.inc'")
gate_model.append("\n")
gate_model.append(r".include '45nm\models\hspice\tran_models\models_nom\NMOS_VTL.inc'")
gate_model.append("\n\n")
for required_gate in required_gates:
    with open(Nangate_path, "r") as file:
        for line in file:
            if "Cellname" in line:
                if " "+required_gate+"_X1." in line:
                    # print(line)
                    l = file.readline()
                    while ((".SUBCKT") not in l):
                        l = file.readline()
                        # print(l)
                    while ((".ENDS") not in l):
                        # print(l.replace(required_gate+"_X1.", required_gate+"_X1_new."))
                        gate_model.append(l.replace(required_gate+"_X1", required_gate+"_X1_new"))
                        l = file.readline()
                    gate_model.append(l)
                    gate_model.append("\n")

In [13]:
## creating the input sources 
lines = []

lines.append("****** Circuit Topology ******\n")
lines.append("V1_supply vcc 0 DC vdd1\n")
for inp in PIs:
    ## depending on the input type
    # print(str(PIs.index(inp)))
    lines.append("vinp"+str(PIs.index(inp))+" "+inp.strip(" ")+" vcc 0 DC vdd1 \n")
## netlist connections
lines.append("\n")
for g in net:
    if (g.numOfInp == 1):
        lines.append("Xu_"+g.name+"_"+str(g.gateNum)+" 0 vcc "+g.inputA1+" "+g.outputZN+" "+g.name+"_X1_new \n")
    elif (g.numOfInp == 2):
        lines.append("Xu_"+g.name+"_"+str(g.gateNum)+" 0 vcc "+g.inputA1+" "+g.inputA2+" "+g.outputZN+" "+g.name+"_X1_new \n")
    elif (g.numOfInp == 3):
        lines.append("Xu_"+g.name+"_"+str(g.gateNum)+" 0 vcc "+g.inputA1+" "+g.inputA2+" "+g.inputA3+" "+g.outputZN+" "+g.name+"_X1_new \n")
    elif (g.numOfInp == 4):
        lines.append("Xu_"+g.name+"_"+str(g.gateNum)+" 0 vcc "+g.inputA1+" "+g.inputA2+" "+g.inputA3+" "+g.inputA4+" "+g.outputZN+" "+g.name+"_X1_new \n")
        
## ouput loads
lines.append("\n")
for outp in POs:
    lines.append("c"+str(POs.index(outp)+1)+" "+outp.strip(" ")+" 0 "+str(outLoad)+"f\n")


In [14]:
## MOSRA Anlysis

mosra_lines = []
mosra_lines.append("\n")
mosra_lines.append("****** MOSRA Analysis ******\n")
mosra_lines.append(".model p1_ra mosra level=1\n")
mosra_lines.append("+tit0 = 5e-8 titfd = 7.5e-10 tittd = 1.45e-20 tn = 0.25\n")
mosra_lines.append(".appendmodel p1_ra mosra PMOS_VTL PMOS\n")
mosra_lines.append(".mosra reltotaltime="+str(reltotaltime)+" RelMode = 2 RelStep = "+str(RelStep)+"\n")

In [15]:
## MOSRA prints
mosra_print_lines = []
mosra_print_lines.append("\n")
mosra_print_lines.append("****** MOSRA delvth prints ******\n")
mosra_print_lines.append("* mosraprint prints IDS/DVTH and dvth\n")
for g in net:
    curr_gate = "Xu_NAND"+str(g.gateNum)+"_1"
    PMOS_num = "M_M2"
    mosra_print_lines.append(".MOSRAPRINT "+curr_gate+"_"+PMOS_num+" dvth("+curr_gate+"."+PMOS_num+",  vds=1, vgs=1, vbs=0)\n")
    PMOS_num = "M_M3"
    mosra_print_lines.append(".MOSRAPRINT "+curr_gate+"_"+PMOS_num+" dvth("+curr_gate+"."+PMOS_num+",  vds=1, vgs=1, vbs=0)\n")
    mosra_print_lines.append("\n")

In [16]:
## Measurements: based on worst path
measurements = []

measurements.append("\n****** Measurements ******\n")
measurements.append(".measure tran TDR  trig v("+worstPathPI+")  td=3n     val=0.5*vdd1 fall=1 targ v("+worstPathPO+") val=0.5*vdd1 rise=1\n")
measurements.append(".measure tran TDF  trig v("+worstPathPI+")  td=3n     val=0.5*vdd1 rise=1 targ v("+worstPathPO+") val=0.5*vdd1 fall=1\n")


In [17]:
## Transient & Params

tranParams = []
tranParams.append("\n****** Transient & Params ******\n")
tranParams.append(".param vdd1="+str(vdd1)+" temp="+str(temp)+" t="+str(t)+"p\n")
tranParams.append(".tran "+resolution+" "+simTime+"\n.options post=2\n")
tranParams.append(".op\n.end\n")

In [18]:
directory_path = benchmark_name+"/HSPICE/"
os.makedirs(directory_path, exist_ok=True)
output_file_path = os.path.join(directory_path+benchmark_name+".sp")
with open(output_file_path, "w") as file:
    file.writelines(gate_model)
    file.writelines(lines)
    file.writelines(mosra_lines)
    # file.writelines(mosra_print_lines)
    file.writelines(measurements)
    file.writelines(tranParams)
file.close()

In [19]:
## systemC output
sysC_H = []
for required_gate in required_gates:
    sysC_H.append("#include \""+required_gate+"_X1.h\"\n")
    
sysC_H.append("SC_MODULE(powerGatesNetlist)\n{\n")
sysC_H.append("\tsc_in <tr_logic> ")
for PI_idx in range(len(PIs)-1):
    sysC_H.append(PIs[PI_idx]+", ")
sysC_H.append(PIs[len(PIs)-1]+";\n")

sysC_H.append("\tsc_out <tr_logic> ")
for PO_idx in range(len(POs)-1):
    sysC_H.append(POs[PO_idx]+", ")
sysC_H.append(POs[len(POs)-1]+";\n")

sysC_H.append("\tsc_signal <tr_logic> ")
for wire_idx in range(len(wires)-1):
    if((not(wires[wire_idx] in PIs)) and (not(wires[wire_idx] in POs))):
        sysC_H.append(wires[wire_idx]+", ")
sysC_H.append(wires[len(wires)-1]+";\n")

sysC_H.append("\n\tint numOfGates;\n\tdouble t;\n\tdouble outLoad;\n\n")

i = 1
for g in net:
    sysC_H.append("\t"+g.name+"_X1* "+g.name+"_Gate"+str(i)+";\n")
    i = i+1

sysC_H.append("\n\tSC_CTOR(powerGatesNetlist)\n\t{\n\t\tnumOfGates = "+str(len(net))+";\n\t\tt = "+str(reltotaltime)+";\n\t\toutLoad = "+str(outLoad)+";")

i = 1
for g in net:
    sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+" = new "+g.name+"_X1(\""+g.name+"_instance"+str(g.gateNum)+"\");\n")
    if (g.numOfInp == 1):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A1"+"("+g.inputA1+");\n")
    if (g.numOfInp == 2):
        if (g.name.startswith("XOR")): ## XOR has only 2
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A"+"("+g.inputA1+");\n")
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->B"+"("+g.inputA2+");\n")
        else:    
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A1"+"("+g.inputA1+");\n")
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A2"+"("+g.inputA2+");\n")
    if (g.numOfInp == 3):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A1"+"("+g.inputA1+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A2"+"("+g.inputA2+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A3"+"("+g.inputA3+");\n")
    if (g.numOfInp == 4):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A1"+"("+g.inputA1+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A2"+"("+g.inputA2+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A3"+"("+g.inputA3+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A4"+"("+g.inputA4+");\n")
    sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->ZN"+"("+g.outputZN+");\n")
    if (g.outputZN in POs):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->load_c"+" = outLoad;\n")
    else:
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->load_c = 0;\n")
    sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->aged_time = t;\n\n")
    i = i+1

sysC_H.append("\t\tcout << \"all gates are instantiated \" << numberOfGates << \"\\n\";\n\t\tSC_THREAD(setupSim);\n\t}\n\tvoid ini();\n\tvoid setupSim();\n")

j = 1
for required_gate_idx in range(len(required_gates)):
    sysC_H.append("\tvoid notifyAlfaCalc"+required_gates[required_gate_idx]+"(")
    sysC_H.append(required_gates[required_gate_idx]+"_X1 *gate"+");\n")
    # sysC_H.append(required_gates[len(required_gates)-1]+"_X1 *gate"+str(j)+");\n};\n")
    j=j+1
sysC_H.append("\n};\n")


directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_output_file_path = os.path.join(directory_path+"powerGatesNetlist.h")
with open(sysC_output_file_path, "w") as file:
    file.writelines(sysC_H)
file.close()

In [20]:
sysC_C = []
sysC_C.append("#include \"powerGatesNetlist.h\"\n\nvoid powerGatesNetlist::ini()\n{\n\n}\n\n")
sysC_C.append("void powerGatesNetlist::setupSim()\n{\n\twhile (true)\n\t{\n")
sysC_C.append("\t\tfor (int i = 0; i < numOfGates; i++){\n")
k = 1
for g in net:
    sysC_C.append("\t\t\tnotifyAlfaCalc"+g.name+"("+g.name+"_Gate"+str(k)+");\n")
    k= k+1
sysC_C.append("\t\t}\n\t\twait();\n\t}\n}\n\n")

j = 1
for required_gate_idx in range(len(required_gates)):
    sysC_C.append("void powerGatesNetlist::notifyAlfaCalc"+required_gates[required_gate_idx]+"(")
    sysC_C.append(required_gates[required_gate_idx]+"_X1 *gate"+")\n")
    sysC_C.append("{\n\tif (gate->ZN.read().alfa == -1){\n\t\tgate->findAlfa.notify();\n\t\twait(0, SC_NS);\n\t\tcout << \"CHECK ALFA ZN gate \" << gate->ZN.read().gateNumOut << \" \" << gate->ZN.read().alfa << \"\\n\";\n\t}\n}\n")
    # sysC_C.append(required_gates[len(required_gates)-1]+"_X1 *gate"+str(j)+")\n")


directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_C_output_file_path = os.path.join(directory_path+"powerGatesNetlist.cpp")
with open(sysC_C_output_file_path, "w") as file:
    file.writelines(sysC_C)
file.close()

In [21]:

sysC_TB_H = []
sysC_TB_H.append("#include \"powerGatesNetlist.h\"\nSC_MODULE(powerGatesNetlistTB)\n{\n\n")
sysC_TB_H.append("\tsc_signal <tr_logic> ")
for PI_idx in range(len(PIs)-1):
    sysC_TB_H.append("testData"+str(PI_idx+1)+", ")
sysC_TB_H.append("testData"+str(len(PIs))+";\n")

sysC_TB_H.append("\tsc_signal <tr_logic> ")
for PO_idx in range(len(POs)-1):
    sysC_TB_H.append("testRes"+str(PO_idx+1)+", ")
sysC_TB_H.append("testRes"+str(len(POs))+";\n")

sysC_TB_H.append("\n\tdouble inptr;\n")

sysC_TB_H.append("\n\tsc_signal<sc_logic> reset, clock;\n\tsc_time sprocketRate = sc_time(102, SC_NS);\n\tpowerGatesNetlist* UUT;\n\n")

sysC_TB_H.append("\tSC_CTOR(powerGatesNetlistTB)\n\t{\n\n\t\tinptr = "+str(input_transition)+";\n\n\t\tUUT = new powerGatesNetlist(\"powerGatesNetlist_instance\");\n")
p=1
for PI in PIs:
    sysC_TB_H.append("\t\tUUT->"+PI.strip(" ")+"(testData"+str(p)+");\n")
    p=p+1

o=1
for PO in POs:
    sysC_TB_H.append("\t\tUUT->"+PO.strip(" ")+"(testRes"+str(o)+");\n")
    o=o+1

sysC_TB_H.append("\n\t\tSC_THREAD(clockGeneration);\n\t\tSC_THREAD(resetAssertion);\n")
for k in range(len(PIs)):
    sysC_TB_H.append("\t\tSC_THREAD(testData"+str(k+1)+"Waveform);\n")

sysC_TB_H.append("\n\t}\n\tvoid clockGeneration();\n\tvoid resetAssertion();\n")
for k in range(len(PIs)):
    sysC_TB_H.append("\tvoid testData"+str(k+1)+"Waveform();\n")

sysC_TB_H.append("};\n")




directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_TB_H_output_file_path = os.path.join(directory_path+"netlistSimulationTB.h")
with open(sysC_TB_H_output_file_path, "w") as file:
    file.writelines(sysC_TB_H)
file.close()

In [22]:
sysC_TB_C = []
sysC_TB_C.append("#include \"netlistSimulationTB.h\"\nvoid powerGatesNetlistTB::clockGeneration()\n{\n\twhile (true)\n\t{\n\t\twait(17, SC_NS);\n\t\tclock = SC_LOGIC_0;\n\t\twait(17, SC_NS);\n\t\tclock = SC_LOGIC_1;\n\t}\n}\n")
sysC_TB_C.append("void powerGatesNetlistTB::resetAssertion()\n{\n\twhile (true)\n\t{\n\t\twait(37, SC_NS);\n\t\treset = SC_LOGIC_0;\n\t\twait(59, SC_NS);\n\t\treset = SC_LOGIC_1;\n\t\twait(59, SC_NS);\n\t\treset = SC_LOGIC_0;\n\t\twait();\n\t}\n}")

m=1
for PI in PIs:
    sysC_TB_C.append("void powerGatesNetlistTB::testData"+str(m)+"Waveform()\n{\n\twhile (true)\n{\n")
    sysC_TB_C.append("\t\ttestData"+str(m)+".write({ SC_LOGIC_0, inptr , inptr , 0.5 });\n\t\twait(1000, SC_NS);\n\t\ttestData"+str(m)+".write({ SC_LOGIC_1, inptr , inptr , 0.5 });\n\t\twait(15000, SC_NS);\n\t\twait();\n\t}\n}\n")
    m=m+1




directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_TB_C_output_file_path = os.path.join(directory_path+"netlistSimulationTB.cpp") 
with open(sysC_TB_C_output_file_path, "w") as file:
    file.writelines(sysC_TB_C)
file.close()

In [23]:
sysC_sim_C = []
sysC_sim_C.append("#include \"netlistSimulationTB.h\"\n\nint sc_main(int argc, char** argv)\n{\n\tpowerGatesNetlistTB* TOP = new powerGatesNetlistTB(\"netlistSimulationTB_instance\");")
sysC_sim_C.append("\n\tclock_t start, end;\n\tstart = clock();\n\tsc_start(400000, SC_NS);\n\tend = clock();\n\tdouble time_taken = double(end - start) / double(CLOCKS_PER_SEC);\n\tcout << \"Time taken by program is : \" << fixed << time_taken;\n\tcout << \" sec \" << endl;\n\treturn 0;\n}")


directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_sim_C_output_file_path = os.path.join(directory_path+"netlistSimulation.cpp") 
with open(sysC_sim_C_output_file_path, "w") as file:
    file.writelines(sysC_sim_C)
file.close()